# max-back-tied-half — worked example 3: Apply maximum_back with Broadcast Inputs and unbroadcast

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `max-back-tied-half`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

When `maximum(x, y)` involves broadcasting — for instance `x: (3,)` and `y: (1,)` — the backward function must apply `unbroadcast` to restore each gradient to the shape of its original input. Without unbroadcasting, the gradient w.r.t. the smaller operand would have the wrong shape. The `unbroadcast` operation sums over the axes that were broadcast during the forward pass.

## Worked solution

**Step 1 — define unbroadcast.**
Unbroadcasting sums over any leading axes and over any axes where the original tensor had size 1. This reverses the broadcast expansion.

**Step 2 — compute maximum_back0 with unbroadcast.**
We apply the half-mass mask to `grad_out * bool_sum_x`, then call `unbroadcast` so the result has the same shape as `x`.

**Step 3 — demonstrate with x: (4,) and y: scalar.**
When `y` is scalar, `grad_out` has the same shape as `x`, but the gradient w.r.t. `y` must be summed down to a scalar.

**Step 4 — verify both gradients against autograd.**
We compute the same operation with `requires_grad=True` and `.backward()` to confirm shapes and values.

In [ ]:
import torch as t

def unbroadcast(grad, original):
    """Sum grad over axes that were broadcast to match original's shape."""
    # Sum leading dims if grad has more dims than original
    ndim_diff = grad.ndim - original.ndim
    if ndim_diff > 0:
        grad = grad.sum(tuple(range(ndim_diff)))
    # Sum over axes where original had size 1
    for i, (og, gg) in enumerate(zip(original.shape, grad.shape)):
        if og == 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def maximum_back0_broadcast(grad_out, x, y):
    mask = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return unbroadcast(grad_out * mask, x)

def maximum_back1_broadcast(grad_out, x, y):
    mask = (x < y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return unbroadcast(grad_out * mask, y)

# x: (4,) broadcast with y: scalar tensor (1,)
t.manual_seed(61)
x = t.tensor([3.0, 1.0, 2.0, -1.0])
y = t.tensor([2.0])     # scalar broadcast
grad_out = t.ones(4)

g0 = maximum_back0_broadcast(grad_out, x, y)
g1 = maximum_back1_broadcast(grad_out, x, y)

print(f"x: {x.tolist()}, y: {y.tolist()}")
print(f"g0 (dL/dx) shape: {g0.shape}  expected (4,)")
print(f"g1 (dL/dy) shape: {g1.shape}  expected (1,)")
print(f"g0: {g0.tolist()}")
print(f"g1: {g1.tolist()}")

# Verify against autograd
x_ag = x.clone().requires_grad_(True)
y_ag = y.clone().requires_grad_(True)
out_ag = t.maximum(x_ag, y_ag)
loss = (out_ag * grad_out).sum()
loss.backward()
print(f"\ng0 match: {t.allclose(g0, x_ag.grad)}")
print(f"g1 match: {t.allclose(g1, y_ag.grad)}")